# Modal single-GPU training preflight

This experiment exercises the current Diffusion-ARC single-GPU path on one Modal H200. It uses the already populated Modal Volume and performs no model or dataset download.

A canonical PASS requires the exact pinned source commit, frozen NVARC and VAE parameters, FP32 FLUX parameters and AdamW state, BF16 FLUX compute, one finite backward pass, one real optimizer update, and a non-zero FLUX parameter change.

Before running, select one H200, attach the diffusion-arc-models Volume at /mnt/diffusion-arc-models with write access, and attach a Modal Secret whose key is GITHUB_TOKEN. Start from a fresh kernel and run top-to-bottom.

The source clone and synthetic dataset are intentionally ephemeral. Checkpoints and JSON reports remain durable in the attached Volume.


## Experiment and iteration plan

The notebook first loads all models and creates one reusable physical FLUX microbatch. The preview cell can then run forward and backward repeatedly without calling optimizer.step or changing weights. The guarded final cell reconstructs the complete input path and performs exactly one AdamW update.

For a compatible forward or loop change, commit and push locally, replace SOURCE_COMMIT, rerun source sync, and reload the hot-path modules. A canonical final report requires model loading, microbatch construction, and execution to use the same commit; after exploratory changes, restart and run top-to-bottom.


## 1. Fix experiment identity and validate durable storage


In [ ]:
from __future__ import annotations

import base64
from datetime import datetime, timezone
import hashlib
import json
import os
from pathlib import Path
import re

SEED = 167
NOTEBOOK_PROCEDURE_VERSION = 1
SOURCE_COMMIT = '4cb97551b1bbafe7f25745d4567c6acdec37a80e'
REPOSITORY_URL = 'https://github.com/joshuakauling/diffusion-arc.git'
SOURCE_DIR = Path('/root/diffusion-arc')
MINIMUM_VRAM_GIB = 120.0
EXPECTED_FLUX_REPO_ID = 'black-forest-labs/FLUX.2-klein-base-4B'
EXPECTED_FLUX_REVISION = (
    '8c44a2fbef88fae175da65df054db0f901aa9747'
)
FLUX_REVISION_METADATA_URL = (
    'https://huggingface.co/api/models/black-forest-labs/'
    'FLUX.2-klein-base-4B/revision/'
    f'{EXPECTED_FLUX_REVISION}?blobs=true'
)
EXPECTED_NVARC_HANDLE = (
    'sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1'
)
NVARC_VERSION_URL = (
    'https://www.kaggle.com/models/sorokin/qwen3_4b_grids15_sft139/'
    'transformers/bfloat16/1'
)
EXPECTED_PROJECT_SHA256 = {
    'flux_transformer': (
        '9c5fed22b76baea749d88fc2abe3ad53245e7b21a0d353a762665eea00043b92'
    ),
    'diffusers_vae': (
        'ca70d2202afe6415bdbcb8793ba8cd99fd159cfe6192381504d6c4d3036e0f04'
    ),
}
EXPECTED_FLUX_LOADER_SHA256 = {
    (
        'flux2-klein-base-4b/transformer/'
        'diffusion_pytorch_model.safetensors'
    ): 'e109674697ffa1a3983126e32512f5428a9442bd8df59f9c95566ee90a473bb6',
    (
        'flux2-klein-base-4b/vae/'
        'diffusion_pytorch_model.safetensors'
    ): EXPECTED_PROJECT_SHA256['diffusers_vae'],
}
EXPECTED_NVARC_FILE_SIZES = {
    'model-00001-of-00002.safetensors': 4_996_836_472,
    'model-00002-of-00002.safetensors': 2_270_397_024,
}

VOLUME_ROOT = Path('/mnt/diffusion-arc-models')
FLUX_CHECKPOINT = VOLUME_ROOT / 'flux2-klein-base-4b'
NVARC_CHECKPOINT = VOLUME_ROOT / 'nvarc-qwen3-4b-grids15-sft139'
DOWNLOAD_MANIFEST = VOLUME_ROOT / 'download-manifest.json'
RESULTS_ROOT = VOLUME_ROOT / 'results' / 'single-gpu-training-preflight'

if re.fullmatch(r'[0-9a-f]{40}', SOURCE_COMMIT) is None:
    raise ValueError('SOURCE_COMMIT must be a full 40-character Git SHA')
if not VOLUME_ROOT.is_dir():
    raise RuntimeError(f'Attach the model Volume at {VOLUME_ROOT} first')

write_probe = VOLUME_ROOT / '.single-gpu-preflight-write-probe'
try:
    write_probe.write_text('ok\n', encoding='utf-8')
    write_probe.unlink()
except OSError as error:
    raise RuntimeError(f'Modal Volume is not writable: {VOLUME_ROOT}') from error

RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
REPORT_DIR = RESULTS_ROOT / RUN_ID
REPORT_PATH = REPORT_DIR / 'report.json'
REPORT_DIR.mkdir(parents=True, exist_ok=False)

REPORT: dict[str, object] = {
    'schema_version': 1,
    'status': 'RUNNING',
    'run_id': RUN_ID,
    'notebook_procedure_version': NOTEBOOK_PROCEDURE_VERSION,
    'question': (
        'Can the current complete single-GPU path execute one numerically '
        'valid FP32-parameter/BF16-compute AdamW update on an H200?'
    ),
    'configuration': {
        'seed': SEED,
        'minimum_vram_gib': MINIMUM_VRAM_GIB,
        'optimizer_steps': 1,
        'optimizer_batch_size': 1,
        'flux_microbatch_size': 1,
    },
    'paths': {
        'volume_root': str(VOLUME_ROOT),
        'flux_checkpoint': str(FLUX_CHECKPOINT),
        'nvarc_checkpoint': str(NVARC_CHECKPOINT),
        'download_manifest': str(DOWNLOAD_MANIFEST),
        'report': str(REPORT_PATH),
    },
    'failures': [],
    'previews': [],
}


def save_report() -> None:
    temporary_path = REPORT_PATH.with_suffix('.json.tmp')
    temporary_path.write_text(
        json.dumps(REPORT, indent=2, sort_keys=True, allow_nan=False) + '\n',
        encoding='utf-8',
    )
    temporary_path.replace(REPORT_PATH)
    if hasattr(os, 'sync'):
        os.sync()


def record_failure(stage: str, error: BaseException) -> None:
    failure: dict[str, object] = {
        'stage': stage,
        'type': type(error).__name__,
        'message': str(error),
        'recorded_at_utc': datetime.now(timezone.utc).isoformat(),
    }
    memory_probe = globals().get('safe_cuda_memory_snapshot')
    if callable(memory_probe):
        failure['cuda_memory'] = memory_probe(f'failure:{stage}')
    failures = REPORT.setdefault('failures', [])
    if isinstance(failures, list):
        failures.append(failure)
    REPORT['status'] = 'FAIL'
    REPORT['failed_stage'] = stage
    REPORT['error'] = {
        'type': type(error).__name__,
        'message': str(error),
    }
    save_report()


required_paths = {
    'FLUX checkpoint': FLUX_CHECKPOINT,
    'NVARC checkpoint': NVARC_CHECKPOINT,
    'download manifest': DOWNLOAD_MANIFEST,
}
missing_paths = {
    name: str(path) for name, path in required_paths.items() if not path.exists()
}
if missing_paths:
    error = FileNotFoundError(f'Missing model artifacts: {missing_paths}')
    record_failure('volume-artifacts', error)
    raise error

try:
    manifest_bytes = DOWNLOAD_MANIFEST.read_bytes()
    manifest_payload = json.loads(manifest_bytes)
    manifest_paths = {
        'flux': Path(manifest_payload['flux']['path']).resolve(),
        'nvarc': Path(manifest_payload['nvarc']['path']).resolve(),
    }
    expected_manifest_paths = {
        'flux': FLUX_CHECKPOINT.resolve(),
        'nvarc': NVARC_CHECKPOINT.resolve(),
    }
    if manifest_paths != expected_manifest_paths:
        raise RuntimeError(
            f'Download manifest paths differ: {manifest_paths}'
        )
    manifest_contract = {
        'flux_repo_id': (
            manifest_payload['flux']['repo_id']
            == EXPECTED_FLUX_REPO_ID
        ),
        'flux_requested_revision': (
            manifest_payload['flux']['requested_revision']
            == EXPECTED_FLUX_REVISION
        ),
        'flux_resolved_revision': (
            manifest_payload['flux']['resolved_revision']
            == EXPECTED_FLUX_REVISION
        ),
        'project_file_sha256': (
            manifest_payload['flux']['project_file_sha256']
            == EXPECTED_PROJECT_SHA256
        ),
        'nvarc_handle': (
            manifest_payload['nvarc']['kaggle_handle']
            == EXPECTED_NVARC_HANDLE
        ),
        'nvarc_version': manifest_payload['nvarc']['version'] == 1,
        'nvarc_weight_sizes': (
            manifest_payload['nvarc']['weight_file_bytes']
            == EXPECTED_NVARC_FILE_SIZES
        ),
    }
    failed_manifest_checks = [
        name for name, passed in manifest_contract.items() if not passed
    ]
    if failed_manifest_checks:
        raise RuntimeError(
            f'Download manifest checks failed: {failed_manifest_checks}'
        )
except Exception as error:
    record_failure('download-manifest', error)
    raise

REPORT['download_manifest'] = {
    'sha256': hashlib.sha256(manifest_bytes).hexdigest(),
    'payload': manifest_payload,
    'checks': manifest_contract,
}

os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

save_report()
print(f'Run ID: {RUN_ID}')
print(f'Durable report: {REPORT_PATH}')
print('Persistent model artifacts: present')


## 2. Fetch the exact source commit

The required GITHUB_TOKEN is passed as a temporary Git Basic-auth header and is never placed in a URL, file, output, or report. Source refresh refuses to overwrite tracked edits inside the ephemeral clone.


In [ ]:
import subprocess


def run_command(
    label: str,
    command: list[str],
    *,
    cwd: Path | None = None,
    env: dict[str, str] | None = None,
) -> str:
    completed = subprocess.run(
        command,
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )
    if completed.returncode != 0:
        tail = '\n'.join(completed.stdout.splitlines()[-40:])
        if tail:
            print(tail)
        raise RuntimeError(f'{label} failed with exit code {completed.returncode}')
    return completed.stdout.strip()


def current_source_head() -> str:
    return run_command(
        'resolve source HEAD', ['git', 'rev-parse', 'HEAD'], cwd=SOURCE_DIR
    )


def authenticated_git_environment() -> dict[str, str]:
    github_token = os.environ.get('GITHUB_TOKEN', '').strip()
    if not github_token:
        raise RuntimeError(
            "Modal Secret key 'GITHUB_TOKEN' is missing or empty. "
            'Attach the Secret and restart the kernel.'
        )
    basic_credential = base64.b64encode(
        f'x-access-token:{github_token}'.encode('utf-8')
    ).decode('ascii')
    environment = os.environ.copy()
    environment.pop('GITHUB_TOKEN', None)
    environment['GIT_TERMINAL_PROMPT'] = '0'
    environment['GIT_CONFIG_COUNT'] = '1'
    environment['GIT_CONFIG_KEY_0'] = (
        'http.https://github.com/.extraheader'
    )
    environment['GIT_CONFIG_VALUE_0'] = (
        f'AUTHORIZATION: basic {basic_credential}'
    )
    return environment


try:
    git_environment = authenticated_git_environment()
except Exception as error:
    record_failure('source-authentication', error)
    raise

try:
    if SOURCE_DIR.exists() and not (SOURCE_DIR / '.git').is_dir():
        raise RuntimeError(f'{SOURCE_DIR} exists but is not the expected clone')
    if not SOURCE_DIR.exists():
        SOURCE_DIR.mkdir(parents=True)
        run_command('git init', ['git', 'init'], cwd=SOURCE_DIR)

    remotes = run_command(
        'list Git remotes', ['git', 'remote'], cwd=SOURCE_DIR
    ).splitlines()
    if 'origin' not in remotes:
        run_command(
            'add Git origin',
            ['git', 'remote', 'add', 'origin', REPOSITORY_URL],
            cwd=SOURCE_DIR,
        )
    origin_url = run_command(
        'read Git origin', ['git', 'remote', 'get-url', 'origin'], cwd=SOURCE_DIR
    )
    if origin_url != REPOSITORY_URL:
        raise RuntimeError(f'Unexpected Git origin: {origin_url}')

    tracked_changes = run_command(
        'inspect source worktree',
        ['git', 'status', '--porcelain', '--untracked-files=no'],
        cwd=SOURCE_DIR,
    )
    if tracked_changes:
        raise RuntimeError('Preserve tracked source edits before refreshing')

    run_command(
        'fetch pinned source',
        ['git', 'fetch', '--depth=1', 'origin', SOURCE_COMMIT],
        cwd=SOURCE_DIR,
        env=git_environment,
    )
    fetched_commit = run_command(
        'resolve fetched source', ['git', 'rev-parse', 'FETCH_HEAD'], cwd=SOURCE_DIR
    )
    if fetched_commit != SOURCE_COMMIT:
        raise RuntimeError(f'Fetched source differs: {fetched_commit}')
    run_command(
        'checkout pinned source',
        ['git', 'checkout', '--detach', 'FETCH_HEAD'],
        cwd=SOURCE_DIR,
    )
    RESOLVED_SOURCE_COMMIT = run_command(
        'resolve source HEAD', ['git', 'rev-parse', 'HEAD'], cwd=SOURCE_DIR
    )
    if RESOLVED_SOURCE_COMMIT != SOURCE_COMMIT:
        raise RuntimeError('Checked-out source differs from SOURCE_COMMIT')
except Exception as error:
    record_failure('source-fetch', error)
    raise
finally:
    git_environment.clear()
    del git_environment

REPORT['source'] = {
    'repository': REPOSITORY_URL,
    'requested_commit': SOURCE_COMMIT,
    'resolved_commit': RESOLVED_SOURCE_COMMIT,
    'path': str(SOURCE_DIR),
    'authentication': 'github_pat_basic_header',
}
save_report()
print(f'Source ready at {RESOLVED_SOURCE_COMMIT[:12]} in {SOURCE_DIR}')


## 3. Install pinned dependencies and editable source

The cell checks versions before importing PyTorch, changes only mismatched packages, and installs the clone in editable mode without dependency resolution. Restart if a mismatched PyTorch was already imported.


In [ ]:
import importlib
import importlib.metadata
import sys

PINNED_PACKAGES = {
    'accelerate': '1.12.0',
    'diffusers': '0.39.0',
    'einops': '0.8.1',
    'Pillow': '11.3.0',
    'safetensors': '0.8.0',
    'torch': '2.13.0',
    'transformers': '4.56.1',
}


def installed_version(distribution: str) -> str | None:
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None


requirements = [
    f'{name}=={version}'
    for name, version in PINNED_PACKAGES.items()
    if installed_version(name) != version
]
if any(value.startswith('torch==') for value in requirements) and 'torch' in sys.modules:
    raise RuntimeError('Restart: a mismatched torch is already imported')

try:
    if requirements:
        print('Installing:', ' '.join(requirements))
        run_command(
            'install dependencies',
            [
                sys.executable,
                '-m',
                'pip',
                'install',
                '--quiet',
                '--disable-pip-version-check',
                '--no-input',
                '--no-cache-dir',
                *requirements,
            ],
        )
    run_command(
        'install Diffusion-ARC source',
        [
            sys.executable,
            '-m',
            'pip',
            'install',
            '--quiet',
            '--disable-pip-version-check',
            '--no-input',
            '--no-deps',
            '--no-build-isolation',
            '-e',
            str(SOURCE_DIR),
        ],
    )
except Exception as error:
    record_failure('dependency-install', error)
    raise

source_path = str((SOURCE_DIR / 'src').resolve())
if source_path not in sys.path:
    sys.path.insert(0, source_path)
importlib.invalidate_caches()
import diffusion_arc

package_path = Path(diffusion_arc.__file__).resolve()
if SOURCE_DIR.resolve() not in package_path.parents:
    error = RuntimeError(f'Unexpected diffusion_arc import: {package_path}')
    record_failure('source-import', error)
    raise error

INSTALLED_VERSIONS = {
    name: installed_version(name) for name in PINNED_PACKAGES
}
version_mismatches = {
    name: (expected, INSTALLED_VERSIONS[name])
    for name, expected in PINNED_PACKAGES.items()
    if INSTALLED_VERSIONS[name] != expected
}
if version_mismatches:
    error = RuntimeError(f'Package mismatch: {version_mismatches}')
    record_failure('dependency-contract', error)
    raise error

existing_import_commit = globals().get('KERNEL_IMPORT_COMMIT')
if existing_import_commit is not None and existing_import_commit != RESOLVED_SOURCE_COMMIT:
    raise RuntimeError(
        'The source commit changed after package import. Use hot-path reload '
        'only for previews, or restart for a canonical run.'
    )
KERNEL_IMPORT_COMMIT = RESOLVED_SOURCE_COMMIT
REPORT['packages'] = INSTALLED_VERSIONS
REPORT['source']['package_path'] = str(package_path)
REPORT['source']['kernel_import_commit'] = KERNEL_IMPORT_COMMIT
save_report()
print(f'diffusion_arc: {package_path}')
INSTALLED_VERSIONS


## 4. Run repository unit tests


In [ ]:
import time

test_environment = os.environ.copy()
test_environment['PYTHONDONTWRITEBYTECODE'] = '1'
test_environment['PYTHONPATH'] = str(SOURCE_DIR / 'src')
test_started = time.perf_counter()
test_process = subprocess.run(
    [sys.executable, '-B', '-m', 'unittest', 'discover', '-s', 'tests'],
    cwd=SOURCE_DIR,
    env=test_environment,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    check=False,
)
test_seconds = time.perf_counter() - test_started
test_output_tail = '\n'.join(test_process.stdout.splitlines()[-30:])
print(test_output_tail)
REPORT['unit_tests'] = {
    'returncode': test_process.returncode,
    'seconds': test_seconds,
    'output_tail': test_output_tail,
}
if test_process.returncode != 0:
    error = RuntimeError(f'Unit tests returned {test_process.returncode}')
    record_failure('unit-tests', error)
    raise error
save_report()
print(f'Unit tests: PASS in {test_seconds:.2f}s')


## 5. Establish the H200 CUDA contract

The memory guard rejects an 80 GB GPU before model allocation. Memory values are synchronized and recorded in GiB.


In [ ]:
import platform
import torch

try:
    if importlib.metadata.version('torch') != PINNED_PACKAGES['torch']:
        raise RuntimeError('Imported torch differs from the project pin')
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA is unavailable; select an H200 kernel')
    if not torch.cuda.is_bf16_supported():
        raise RuntimeError('Selected CUDA device does not support BF16')
    DEVICE = torch.device('cuda:0')
    torch.cuda.set_device(DEVICE)
    GPU_PROPERTIES = torch.cuda.get_device_properties(DEVICE)
    if 'H200' not in GPU_PROPERTIES.name.upper():
        raise RuntimeError(f'Select an H200, received {GPU_PROPERTIES.name}')
    GPU_VRAM_GIB = GPU_PROPERTIES.total_memory / 2**30
    if GPU_VRAM_GIB < MINIMUM_VRAM_GIB:
        raise RuntimeError(
            f'Requires {MINIMUM_VRAM_GIB:.0f} GiB, received {GPU_VRAM_GIB:.1f}'
        )
except Exception as error:
    record_failure('cuda-contract', error)
    raise


def cuda_memory_snapshot(label: str) -> dict[str, object]:
    torch.cuda.synchronize(DEVICE)
    return {
        'label': label,
        'allocated_gib': torch.cuda.memory_allocated(DEVICE) / 2**30,
        'reserved_gib': torch.cuda.memory_reserved(DEVICE) / 2**30,
        'peak_allocated_gib': torch.cuda.max_memory_allocated(DEVICE) / 2**30,
        'peak_reserved_gib': torch.cuda.max_memory_reserved(DEVICE) / 2**30,
    }


def safe_cuda_memory_snapshot(label: str) -> dict[str, object]:
    try:
        return cuda_memory_snapshot(label)
    except Exception as error:
        return {'label': label, 'unavailable': f'{type(error).__name__}: {error}'}


torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats(DEVICE)
nvidia_smi = run_command(
    'query nvidia-smi',
    [
        'nvidia-smi',
        '--query-gpu=driver_version,name,memory.total',
        '--format=csv,noheader',
    ],
)
CUDA_ENVIRONMENT = {
    'python': platform.python_version(),
    'torch': torch.__version__,
    'cuda_runtime': torch.version.cuda,
    'nvidia_smi': nvidia_smi,
    'gpu_name': GPU_PROPERTIES.name,
    'gpu_vram_gib': GPU_VRAM_GIB,
    'compute_capability': list(torch.cuda.get_device_capability(DEVICE)),
    'bf16_supported': torch.cuda.is_bf16_supported(),
    'sdpa_policy': {
        'flash_enabled': torch.backends.cuda.flash_sdp_enabled(),
        'memory_efficient_enabled': torch.backends.cuda.mem_efficient_sdp_enabled(),
        'math_enabled': torch.backends.cuda.math_sdp_enabled(),
    },
}
REPORT['environment'] = CUDA_ENVIRONMENT
REPORT['memory'] = {'runtime_baseline': cuda_memory_snapshot('runtime-baseline')}
save_report()
print(f'GPU: {GPU_PROPERTIES.name} | VRAM: {GPU_VRAM_GIB:.1f} GiB')
CUDA_ENVIRONMENT


### 5a. Bind the run to exact checkpoint artifacts

This read-only pass hashes the files used by the three production loaders. It checks the FLUX loader weights against official SHA-256 values and rechecks the NVARC handle, version, shard sizes, and observed SHA-256 values. Kaggle does not publish per-file content digests for this version, so the durable report makes that provenance limitation explicit. No model or dataset is downloaded.


In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


provenance_anchors = {
    'flux_transformer': (
        FLUX_CHECKPOINT / 'flux-2-klein-base-4b.safetensors'
    ),
    'diffusers_vae': (
        FLUX_CHECKPOINT / 'vae' / 'diffusion_pytorch_model.safetensors'
    ),
}
loader_roots = {
    'flux_transformer': FLUX_CHECKPOINT / 'transformer',
    'flux_vae': FLUX_CHECKPOINT / 'vae',
    'nvarc': NVARC_CHECKPOINT,
}
try:
    missing_anchors = {
        name: str(path)
        for name, path in provenance_anchors.items()
        if not path.is_file()
    }
    missing_loader_roots = {
        name: str(path)
        for name, path in loader_roots.items()
        if not path.is_dir()
    }
    if missing_anchors or missing_loader_roots:
        raise FileNotFoundError(
            f'Missing checkpoint files: anchors={missing_anchors}, '
            f'loader_roots={missing_loader_roots}'
        )

    nvarc_size_mismatches = {}
    for filename, expected_size in EXPECTED_NVARC_FILE_SIZES.items():
        path = NVARC_CHECKPOINT / filename
        actual_size = path.stat().st_size if path.is_file() else None
        if actual_size != expected_size:
            nvarc_size_mismatches[filename] = {
                'expected': expected_size,
                'actual': actual_size,
            }
    if nvarc_size_mismatches:
        raise RuntimeError(
            f'NVARC shard sizes differ: {nvarc_size_mismatches}'
        )

    artifact_roles: dict[Path, set[str]] = {}
    for role, root in loader_roots.items():
        for path in root.rglob('*'):
            if path.is_file() and path.suffix in {'.json', '.safetensors'}:
                artifact_roles.setdefault(path, set()).add(role)
    for role, path in provenance_anchors.items():
        artifact_roles.setdefault(path, set()).add(f'{role}_anchor')
    if not artifact_roles:
        raise RuntimeError('No checkpoint artifacts were found')

    identity_started = time.perf_counter()
    artifact_identities = {}
    for path in sorted(artifact_roles, key=lambda item: str(item)):
        relative_path = str(path.relative_to(VOLUME_ROOT))
        artifact_identities[relative_path] = {
            'roles': sorted(artifact_roles[path]),
            'bytes': path.stat().st_size,
            'sha256': sha256_file(path),
        }
    identity_seconds = time.perf_counter() - identity_started

    anchor_hashes = {
        name: artifact_identities[str(path.relative_to(VOLUME_ROOT))]['sha256']
        for name, path in provenance_anchors.items()
    }
    if anchor_hashes != EXPECTED_PROJECT_SHA256:
        raise RuntimeError(f'FLUX provenance hashes differ: {anchor_hashes}')
    flux_loader_hashes = {
        path: artifact_identities[path]['sha256']
        for path in EXPECTED_FLUX_LOADER_SHA256
    }
    if flux_loader_hashes != EXPECTED_FLUX_LOADER_SHA256:
        raise RuntimeError(f'FLUX loader hashes differ: {flux_loader_hashes}')
except Exception as error:
    record_failure('checkpoint-identities', error)
    raise

REPORT['checkpoint_identities'] = {
    'seconds': identity_seconds,
    'file_count': len(artifact_identities),
    'verification': {
        'flux': 'official_sha256',
        'flux_metadata_url': FLUX_REVISION_METADATA_URL,
        'nvarc': 'kaggle_handle_version_size_and_observed_sha256',
        'nvarc_version_url': NVARC_VERSION_URL,
        'nvarc_limitation': (
            'Kaggle version metadata does not expose per-file digests.'
        ),
    },
    'files': artifact_identities,
}
save_report()
print(
    f'Checkpoint identities: {len(artifact_identities)} files in '
    f'{identity_seconds:.2f}s'
)


## 6. Create a synthetic canonical dataset and strict config

The temporary layout exercises the real config parser, dataset, tokenizer, and DataLoader without depending on ignored ARC files. One demonstration and one labelled query yield exactly one target.


In [ ]:
TEMP_ROOT = Path('/tmp') / f'diffusion-arc-preflight-{RUN_ID}'
DATA_ROOT = TEMP_ROOT / 'data'
SPLIT_ROOT = DATA_ROOT / 'splits' / 'modal-preflight'
RAW_TASK_ROOT = DATA_ROOT / 'raw' / 'synthetic-arc' / 'training'
TRAINING_CONFIG_PATH = TEMP_ROOT / 'configs' / 'single_gpu.json'
TASK_ID = 'synthetic-modal'
AUGMENTATION_KEY = f'{TASK_ID}_ID_0123456789_0,1'
if TEMP_ROOT.exists():
    raise RuntimeError(f'Refusing to reuse temporary directory: {TEMP_ROOT}')


def write_json(path: Path, payload: object) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2) + '\n', encoding='utf-8')


synthetic_task = {
    'train': [
        {'input': [[1, 0], [0, 1]], 'output': [[0, 1], [1, 0]]}
    ],
    'test': [
        {'input': [[2, 2], [0, 2]], 'output': [[2, 0], [2, 2]]}
    ],
}
split_config = {
    'schema_version': 1,
    'generator_version': 1,
    'source': {'dataset': 'synthetic-arc', 'split': 'training'},
    'palette': 'greedy_rgb_maximin_v1',
    'limits': {'max_pairs': 2, 'max_cells': 16, 'max_tokens': 192},
    'task_splits': {'train': 'train.json', 'validation': 'validation.json'},
    'views': {
        'preflight': {
            'task_split': 'train',
            'output': 'augmented_keys/preflight.json',
            'mode': 'identity',
            'exclude': [],
        }
    },
}
run_config = {
    'schema_version': 1,
    'seed': SEED,
    'device': 'cuda:0',
    'models': {
        'nvarc_checkpoint': str(NVARC_CHECKPOINT),
        'flux_checkpoint': str(FLUX_CHECKPOINT),
        'nvarc_dtype': 'bfloat16',
        'vae_dtype': 'float32',
        'flux_parameter_dtype': 'float32',
        'flux_compute_dtype': 'bfloat16',
    },
    'data': {
        'split_directory': str(SPLIT_ROOT),
        'view_name': 'preflight',
        'shuffle': False,
        'num_workers': 0,
    },
    'loop': {
        'max_optimizer_steps': 1,
        'optimizer_batch_size': 1,
        'flux_microbatch_size': 1,
        'min_context_pairs_before_target': 1,
    },
    'optimizer': {
        'learning_rate': 1e-5,
        'beta1': 0.9,
        'beta2': 0.999,
        'epsilon': 1e-8,
        'weight_decay': 0.01,
    },
}
write_json(RAW_TASK_ROOT / f'{TASK_ID}.json', synthetic_task)
write_json(SPLIT_ROOT / 'config.json', split_config)
write_json(SPLIT_ROOT / 'train.json', [TASK_ID])
write_json(SPLIT_ROOT / 'validation.json', [])
write_json(SPLIT_ROOT / 'augmented_keys' / 'preflight.json', [AUGMENTATION_KEY])
write_json(TRAINING_CONFIG_PATH, run_config)
synthetic_identity_payload = {
    'task': synthetic_task,
    'split_config': split_config,
    'train_task_ids': [TASK_ID],
    'validation_task_ids': [],
    'augmentation_keys': [AUGMENTATION_KEY],
}
synthetic_payload_bytes = json.dumps(
    synthetic_identity_payload,
    sort_keys=True,
    separators=(',', ':'),
).encode('utf-8')
synthetic_files = [
    RAW_TASK_ROOT / f'{TASK_ID}.json',
    SPLIT_ROOT / 'config.json',
    SPLIT_ROOT / 'train.json',
    SPLIT_ROOT / 'validation.json',
    SPLIT_ROOT / 'augmented_keys' / 'preflight.json',
    TRAINING_CONFIG_PATH,
]
REPORT['synthetic_data'] = {
    'task_id': TASK_ID,
    'pair_count': 2,
    'augmentation_key': AUGMENTATION_KEY,
    'split_root': str(SPLIT_ROOT),
    'training_config': str(TRAINING_CONFIG_PATH),
    'payload_sha256': hashlib.sha256(synthetic_payload_bytes).hexdigest(),
    'payload': synthetic_identity_payload,
    'file_sha256': {
        str(path.relative_to(TEMP_ROOT)): sha256_file(path)
        for path in synthetic_files
    },
}
REPORT['training_config'] = run_config
save_report()
print(f'Synthetic split: {SPLIT_ROOT}')
print(f'Training config: {TRAINING_CONFIG_PATH}')


## 7. Load and validate all production models

This is the first expensive cell. The project loader uses local-files-only mode and validates tokenizer, architecture, parameter placement, dtypes, and frozen/trainable ownership before returning.


In [ ]:
from diffusion_arc.training.config import load_single_gpu_training_config
from diffusion_arc.training.single_gpu_setup import (
    build_flow_generator,
    build_flux_optimizer,
    build_single_sample_loader,
    load_single_gpu_models,
    prepare_single_gpu_device,
    seed_single_gpu_process,
)


def module_parameter_summary(module: torch.nn.Module) -> dict[str, object]:
    parameter_count = 0
    trainable_count = 0
    devices: set[str] = set()
    dtypes: set[str] = set()
    tensor_count = 0
    for parameter in module.parameters():
        tensor_count += 1
        parameter_count += parameter.numel()
        if parameter.requires_grad:
            trainable_count += parameter.numel()
        devices.add(str(parameter.device))
        dtypes.add(str(parameter.dtype))
    return {
        'parameter_tensors': tensor_count,
        'parameters': parameter_count,
        'trainable_parameters': trainable_count,
        'devices': sorted(devices),
        'dtypes': sorted(dtypes),
    }


if globals().get('models') is not None:
    raise RuntimeError('Models are already loaded; restart instead of loading twice')
if current_source_head() != KERNEL_IMPORT_COMMIT:
    raise RuntimeError('Source changed after import; restart before loading models')
training_config = load_single_gpu_training_config(TRAINING_CONFIG_PATH)
device = prepare_single_gpu_device(training_config)
seed_single_gpu_process(training_config.seed)
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats(device)
try:
    model_load_started = time.perf_counter()
    models = load_single_gpu_models(training_config, device=device)
    torch.cuda.synchronize(device)
    model_load_seconds = time.perf_counter() - model_load_started
except Exception as error:
    record_failure('model-load', error)
    raise

MODEL_LOAD_COMMIT = KERNEL_IMPORT_COMMIT
MODEL_SUMMARIES = {
    'nvarc': module_parameter_summary(models.nvarc_runner),
    'vae': module_parameter_summary(models.vae_encoder),
    'flux': module_parameter_summary(models.transformer),
}
MODEL_CONTRACT_CHECKS = {
    'nvarc_frozen': MODEL_SUMMARIES['nvarc']['trainable_parameters'] == 0,
    'vae_frozen': MODEL_SUMMARIES['vae']['trainable_parameters'] == 0,
    'flux_fully_trainable': (
        MODEL_SUMMARIES['flux']['trainable_parameters']
        == MODEL_SUMMARIES['flux']['parameters']
    ),
    'nvarc_bfloat16': MODEL_SUMMARIES['nvarc']['dtypes'] == ['torch.bfloat16'],
    'vae_float32': MODEL_SUMMARIES['vae']['dtypes'] == ['torch.float32'],
    'flux_float32': MODEL_SUMMARIES['flux']['dtypes'] == ['torch.float32'],
    'flux_compute_bfloat16': models.flux_compute_dtype == torch.bfloat16,
}
failed_model_checks = [
    name for name, passed in MODEL_CONTRACT_CHECKS.items() if not passed
]
REPORT['models'] = {
    'load_seconds': model_load_seconds,
    'source_commit': MODEL_LOAD_COMMIT,
    'summaries': MODEL_SUMMARIES,
    'checks': MODEL_CONTRACT_CHECKS,
}
if failed_model_checks:
    error = RuntimeError(f'Model checks failed: {failed_model_checks}')
    record_failure('model-contract', error)
    raise error

model_load_memory = cuda_memory_snapshot('after-model-load')
REPORT['memory']['after_model_load'] = model_load_memory
save_report()
print(f'Model load: {model_load_seconds:.2f}s')
print(MODEL_CONTRACT_CHECKS)
model_load_memory


## 8. Materialize one sample and one reusable FLUX microbatch

This runs the real dataset, tokenizer, frozen NVARC, and frozen VAE preparation path. The resulting physical microbatch owns its exact conditioning prefix and target latent, so previews need no repeated NVARC or VAE call.


In [ ]:
from diffusion_arc.training.flux_input import prepare_flux_inputs
from diffusion_arc.training.flux_microbatch import build_flux_microbatch

if globals().get('preview_microbatch') is not None:
    raise RuntimeError('Microbatch already exists; restart before rebuilding it')
if current_source_head() != KERNEL_IMPORT_COMMIT:
    raise RuntimeError('Source changed after import; restart before input preparation')
training_samples = build_single_sample_loader(
    training_config,
    tokenizer=models.tokenizer,
)
sample = next(iter(training_samples))
if len(sample.target_grids) != 1:
    raise RuntimeError(
        f'Synthetic sample must expose one target, got {len(sample.target_grids)}'
    )

torch.cuda.reset_peak_memory_stats(device)
try:
    preparation_started = time.perf_counter()
    prepared_inputs = prepare_flux_inputs(
        sample,
        nvarc_runner=models.nvarc_runner,
        vae_encoder=models.vae_encoder,
        min_context_pairs_before_target=(
            training_config.loop.min_context_pairs_before_target
        ),
    )
    if len(prepared_inputs) != 1:
        raise RuntimeError(f'Expected one FLUX input, got {len(prepared_inputs)}')
    preview_microbatch = build_flux_microbatch(prepared_inputs)
    torch.cuda.synchronize(device)
    preparation_seconds = time.perf_counter() - preparation_started
except Exception as error:
    record_failure('input-preparation', error)
    raise

MICROBATCH_BUILD_COMMIT = KERNEL_IMPORT_COMMIT
MICROBATCH_CONTRACT = {
    'sample_key': str(sample.key),
    'source_tokens': len(sample.input_ids),
    'target_count': preview_microbatch.target_count,
    'clean_latents_shape': list(preview_microbatch.clean_latents.shape),
    'clean_latents_dtype': str(preview_microbatch.clean_latents.dtype),
    'text_shape': list(preview_microbatch.text_inputs.encoder_hidden_states.shape),
    'text_dtype': str(preview_microbatch.text_inputs.encoder_hidden_states.dtype),
    'text_mask_shape': list(
        preview_microbatch.text_inputs.text_attention_mask.shape
    ),
    'image_position_ids_shape': list(preview_microbatch.image_position_ids.shape),
}
if MICROBATCH_CONTRACT['clean_latents_shape'] != [1, 900, 128]:
    raise RuntimeError(f'Unexpected latent shape: {MICROBATCH_CONTRACT}')
if MICROBATCH_CONTRACT['text_shape'][-1] != 7680:
    raise RuntimeError(f'Unexpected text width: {MICROBATCH_CONTRACT}')

preview_optimizer = build_flux_optimizer(
    training_config,
    transformer=models.transformer,
)
if preview_optimizer.state:
    raise RuntimeError('AdamW state must be lazy before the first step')

preparation_memory = cuda_memory_snapshot('after-input-preparation')
REPORT['input_preparation'] = {
    'seconds': preparation_seconds,
    'source_commit': MICROBATCH_BUILD_COMMIT,
    'contract': MICROBATCH_CONTRACT,
}
REPORT['memory']['after_input_preparation'] = preparation_memory
save_report()
print(f'Input preparation: {preparation_seconds:.2f}s')
MICROBATCH_CONTRACT


## 9. Reload only the interactive hot path

After fetching a newer compatible commit, rerun this cell to reload flux_forward first and single_gpu second. Model classes, loader policy, and tensor-contract changes require a fresh kernel instead.


In [ ]:
def reload_hot_path_modules() -> tuple[object, object]:
    global flux_forward_module, single_gpu_module, HOT_PATH_COMMIT
    importlib.invalidate_caches()
    flux_forward_module = importlib.reload(
        importlib.import_module('diffusion_arc.training.flux_forward')
    )
    single_gpu_module = importlib.reload(
        importlib.import_module('diffusion_arc.training.single_gpu')
    )
    HOT_PATH_COMMIT = current_source_head()
    return flux_forward_module, single_gpu_module


reload_hot_path_modules()
hot_path_commits = REPORT.setdefault('hot_path_commits', [])
if isinstance(hot_path_commits, list):
    hot_path_commits.append(HOT_PATH_COMMIT)
save_report()
print(f'Hot path loaded from {HOT_PATH_COMMIT[:12]}')


## 10. Re-runnable forward/backward preview without an optimizer step

This interactive cell resets seeds, clears old gradients, computes one loss, backpropagates, checks every gradient, selects a non-zero update probe, and clears gradients again. It never changes FLUX weights and never creates Adam moments.


In [ ]:
if globals().get('FINAL_STEP_ATTEMPTED', False):
    raise RuntimeError('Restart after attempting the final optimizer step')
if preview_optimizer.state:
    raise RuntimeError('Preview optimizer unexpectedly contains Adam state')

preview_optimizer.zero_grad(set_to_none=True)
seed_single_gpu_process(training_config.seed)
preview_generator = build_flow_generator(training_config, device=device)
models.transformer.train()
torch.cuda.reset_peak_memory_stats(device)
AUTOCAST_PROBE: dict[str, dict[str, object]] = {}


def make_projection_probe(name: str):
    def probe(
        _module: torch.nn.Module,
        inputs: tuple[torch.Tensor, ...],
        output: torch.Tensor,
    ) -> None:
        AUTOCAST_PROBE[name] = {
            'autocast_enabled': torch.is_autocast_enabled('cuda'),
            'input_dtype': str(inputs[0].dtype),
            'output_dtype': str(output.dtype),
        }

    return probe


projection_handles = (
    models.transformer.x_embedder.register_forward_hook(
        make_projection_probe('x_embedder')
    ),
    models.transformer.context_embedder.register_forward_hook(
        make_projection_probe('context_embedder')
    ),
)
try:
    preview_started = time.perf_counter()
    preview_loss = flux_forward_module.compute_flux_microbatch_loss(
        models.transformer,
        preview_microbatch,
        compute_dtype=models.flux_compute_dtype,
        generator=preview_generator,
    )
    if set(AUTOCAST_PROBE) != {'x_embedder', 'context_embedder'}:
        raise RuntimeError(f'Incomplete autocast probe: {AUTOCAST_PROBE}')
    if not all(
        record['autocast_enabled']
        and record['output_dtype'] == 'torch.bfloat16'
        for record in AUTOCAST_PROBE.values()
    ):
        raise RuntimeError(f'Real FLUX BF16 autocast was not observed: {AUTOCAST_PROBE}')
    if preview_loss.ndim != 0 or preview_loss.dtype != torch.float32:
        raise RuntimeError(
            f'Preview loss must be scalar FP32, got '
            f'{preview_loss.shape} and {preview_loss.dtype}'
        )
    if not torch.isfinite(preview_loss).item():
        raise RuntimeError('Preview loss is non-finite')
    preview_loss.backward()
    torch.cuda.synchronize(device)
    preview_seconds = time.perf_counter() - preview_started

    gradient_records = [
        (name, parameter, parameter.grad)
        for name, parameter in models.transformer.named_parameters()
        if parameter.grad is not None
    ]
    if not gradient_records:
        raise RuntimeError('Backward produced no FLUX gradients')
    trainable_tensor_count = sum(
        1 for parameter in models.transformer.parameters() if parameter.requires_grad
    )
    if len(gradient_records) != trainable_tensor_count:
        raise RuntimeError(
            f'Only {len(gradient_records)}/{trainable_tensor_count} FLUX '
            'parameter tensors received gradients'
        )
    gradients = [record[2] for record in gradient_records]
    if any(gradient.dtype != torch.float32 for gradient in gradients):
        raise RuntimeError('Every FLUX gradient must use FP32 storage')
    if not all(torch.isfinite(gradient).all().item() for gradient in gradients):
        raise RuntimeError('Backward produced a non-finite gradient')
    if any(
        parameter.grad is not None for parameter in models.nvarc_runner.parameters()
    ):
        raise RuntimeError('NVARC received gradients')
    if any(
        parameter.grad is not None for parameter in models.vae_encoder.parameters()
    ):
        raise RuntimeError('VAE received gradients')

    per_tensor_norms = torch.stack(
        [torch.linalg.vector_norm(gradient.detach()).double() for gradient in gradients]
    )
    preview_gradient_norm = torch.linalg.vector_norm(per_tensor_norms).item()
    tracked = None
    for name, parameter, gradient in gradient_records:
        flat_gradient = gradient.detach().reshape(-1)
        maximum, maximum_index = flat_gradient.max(dim=0)
        minimum, minimum_index = flat_gradient.min(dim=0)
        if abs(minimum.item()) > abs(maximum.item()):
            selected_value = minimum.item()
            selected_index = int(minimum_index.item())
        else:
            selected_value = maximum.item()
            selected_index = int(maximum_index.item())
        if selected_value != 0.0:
            tracked = (name, parameter, selected_index, selected_value)
            break
    if tracked is None:
        raise RuntimeError('No non-zero FLUX gradient element was found')

    (
        TRACKED_PARAMETER_NAME,
        tracked_parameter,
        TRACKED_PARAMETER_INDEX,
        TRACKED_PREVIEW_GRADIENT,
    ) = tracked
    TRACKED_PARAMETER_BEFORE = float(
        tracked_parameter.detach().reshape(-1)[TRACKED_PARAMETER_INDEX].item()
    )
    preview_memory = cuda_memory_snapshot('preview-after-backward')
except Exception as error:
    record_failure('preview-forward-backward', error)
    raise
finally:
    for handle in projection_handles:
        handle.remove()

PREVIEW_COMPLETED = True
PREVIEW_RESULT = {
    'model_load_commit': MODEL_LOAD_COMMIT,
    'microbatch_commit': MICROBATCH_BUILD_COMMIT,
    'hot_path_commit': HOT_PATH_COMMIT,
    'loss': float(preview_loss.detach().item()),
    'loss_dtype': str(preview_loss.dtype),
    'seconds': preview_seconds,
    'gradient_tensors': len(gradient_records),
    'trainable_parameter_tensors': trainable_tensor_count,
    'gradient_dtype': 'torch.float32',
    'gradient_norm': preview_gradient_norm,
    'tracked_parameter': TRACKED_PARAMETER_NAME,
    'tracked_parameter_index': TRACKED_PARAMETER_INDEX,
    'tracked_gradient': TRACKED_PREVIEW_GRADIENT,
    'autocast_probe': AUTOCAST_PROBE,
    'adam_parameter_states': len(preview_optimizer.state),
    'cuda_memory': preview_memory,
}
REPORT['previews'].append(PREVIEW_RESULT)
REPORT['status'] = 'RUNNING'
REPORT.pop('failed_stage', None)
REPORT.pop('error', None)
save_report()
preview_optimizer.zero_grad(set_to_none=True)
del (
    preview_loss,
    gradients,
    gradient_records,
    per_tensor_norms,
    flat_gradient,
    gradient,
    maximum,
    maximum_index,
    minimum,
    minimum_index,
    tracked,
)
torch.cuda.empty_cache()
print('Preview PASS: weights unchanged and Adam state empty')
PREVIEW_RESULT


## 11. Execute exactly one guarded optimizer step

Run this once after a successful preview. It refuses mixed commits, builds fresh AdamW state, reconstructs NVARC and VAE inputs through the real loop, performs one update, and validates every persistent precision boundary. Restart after any attempted final step, successful or not.


In [ ]:
import math

if globals().get('FINAL_STEP_ATTEMPTED', False):
    raise RuntimeError('Final optimizer step was already attempted')
if not globals().get('PREVIEW_COMPLETED', False):
    raise RuntimeError('Run a successful preview first')

canonical_commit = current_source_head()
commit_contract = {
    'current_source': canonical_commit,
    'model_load': MODEL_LOAD_COMMIT,
    'microbatch_build': MICROBATCH_BUILD_COMMIT,
    'hot_path': HOT_PATH_COMMIT,
}
if len(set(commit_contract.values())) != 1:
    raise RuntimeError(
        f'Canonical run requires one commit; restart first: {commit_contract}'
    )
if any(
    parameter.grad is not None for parameter in models.transformer.parameters()
):
    raise RuntimeError('Preview gradients were not fully cleared')

final_optimizer = build_flux_optimizer(
    training_config,
    transformer=models.transformer,
)
if final_optimizer.state:
    raise RuntimeError('Fresh AdamW must have no state')
tracked_parameters = dict(models.transformer.named_parameters())
final_tracked_parameter = tracked_parameters[TRACKED_PARAMETER_NAME]
final_parameter_before = float(
    final_tracked_parameter.detach().reshape(-1)[TRACKED_PARAMETER_INDEX].item()
)
optimizer_step_records: list[dict[str, object]] = []


def capture_optimizer_step(
    loss: torch.Tensor,
    target_count: int,
    microbatch_count: int,
) -> None:
    optimizer_step_records.append(
        {
            'loss': float(loss.item()),
            'loss_dtype': str(loss.dtype),
            'target_count': target_count,
            'microbatch_count': microbatch_count,
        }
    )


optimizer_step_counts = {'attempted': 0, 'completed': 0}


def count_optimizer_step_attempt(_optimizer, _args, _kwargs) -> None:
    optimizer_step_counts['attempted'] += 1


def count_optimizer_step_completion(_optimizer, _args, _kwargs) -> None:
    optimizer_step_counts['completed'] += 1


def optimizer_state_step(state: dict[object, object]) -> int | None:
    value = state.get('step')
    if isinstance(value, torch.Tensor):
        if value.numel() != 1:
            return None
        return int(value.detach().item())
    if isinstance(value, (int, float)):
        return int(value)
    return None


step_pre_hook = final_optimizer.register_step_pre_hook(
    count_optimizer_step_attempt
)
step_post_hook = final_optimizer.register_step_post_hook(
    count_optimizer_step_completion
)
seed_single_gpu_process(training_config.seed)
final_generator = build_flow_generator(training_config, device=device)
torch.cuda.reset_peak_memory_stats(device)
FINAL_STEP_ATTEMPTED = True
try:
    final_step_started = time.perf_counter()
    single_gpu_module.run_single_gpu_training(
        (sample,),
        nvarc_runner=models.nvarc_runner,
        vae_encoder=models.vae_encoder,
        transformer=models.transformer,
        optimizer=final_optimizer,
        flux_compute_dtype=models.flux_compute_dtype,
        max_optimizer_steps=training_config.loop.max_optimizer_steps,
        optimizer_batch_size=training_config.loop.optimizer_batch_size,
        flux_microbatch_size=training_config.loop.flux_microbatch_size,
        min_context_pairs_before_target=(
            training_config.loop.min_context_pairs_before_target
        ),
        generator=final_generator,
        on_optimizer_step=capture_optimizer_step,
    )
    torch.cuda.synchronize(device)
    final_step_seconds = time.perf_counter() - final_step_started
    optimizer_step_memory = cuda_memory_snapshot('optimizer-step-complete')

    final_parameter_after = float(
        final_tracked_parameter.detach()
        .reshape(-1)[TRACKED_PARAMETER_INDEX]
        .item()
    )
    parameter_delta = final_parameter_after - final_parameter_before
    final_gradients = [
        parameter.grad
        for parameter in models.transformer.parameters()
        if parameter.grad is not None
    ]
    trainable_parameter_tensor_count = sum(
        1
        for parameter in models.transformer.parameters()
        if parameter.requires_grad
    )
    moment_tensors = [
        state[name]
        for state in final_optimizer.state.values()
        for name in ('exp_avg', 'exp_avg_sq')
        if isinstance(state.get(name), torch.Tensor)
    ]
    optimizer_state_steps = [
        optimizer_state_step(state)
        for state in final_optimizer.state.values()
    ]
    FINAL_CHECKS = {
        'optimizer_step_attempted_once': (
            optimizer_step_counts['attempted'] == 1
        ),
        'optimizer_step_completed_once': (
            optimizer_step_counts['completed'] == 1
        ),
        'one_optimizer_callback': len(optimizer_step_records) == 1,
        'one_target': len(optimizer_step_records) == 1
        and optimizer_step_records[0]['target_count'] == 1,
        'one_microbatch': len(optimizer_step_records) == 1
        and optimizer_step_records[0]['microbatch_count'] == 1,
        'finite_fp32_loss': len(optimizer_step_records) == 1
        and optimizer_step_records[0]['loss_dtype'] == 'torch.float32'
        and math.isfinite(optimizer_step_records[0]['loss']),
        'real_bf16_autocast_observed': all(
            record['autocast_enabled']
            and record['output_dtype'] == 'torch.bfloat16'
            for record in PREVIEW_RESULT['autocast_probe'].values()
        ),
        'flux_parameters_fp32': all(
            parameter.dtype == torch.float32
            for parameter in models.transformer.parameters()
        ),
        'flux_gradients_exist': bool(final_gradients),
        'all_flux_parameters_have_gradients': (
            len(final_gradients) == trainable_parameter_tensor_count
        ),
        'flux_gradients_fp32': bool(final_gradients)
        and all(
            gradient.dtype == torch.float32 for gradient in final_gradients
        ),
        'flux_gradients_finite': bool(final_gradients)
        and all(
            torch.isfinite(gradient).all().item()
            for gradient in final_gradients
        ),
        'nvarc_gradients_absent': all(
            parameter.grad is None
            for parameter in models.nvarc_runner.parameters()
        ),
        'vae_gradients_absent': all(
            parameter.grad is None
            for parameter in models.vae_encoder.parameters()
        ),
        'adam_state_created': bool(final_optimizer.state),
        'adam_state_steps_equal_one': bool(optimizer_state_steps)
        and len(optimizer_state_steps) == len(final_optimizer.state)
        and all(step == 1 for step in optimizer_state_steps),
        'adam_moments_exist': bool(moment_tensors),
        'adam_moments_fp32': bool(moment_tensors)
        and all(moment.dtype == torch.float32 for moment in moment_tensors),
        'adam_moments_finite': bool(moment_tensors)
        and all(
            torch.isfinite(moment).all().item() for moment in moment_tensors
        ),
        'tracked_parameter_finite': (
            math.isfinite(final_parameter_before)
            and math.isfinite(final_parameter_after)
            and math.isfinite(parameter_delta)
        ),
        'tracked_parameter_changed': parameter_delta != 0.0,
    }
    failed_final_checks = [
        name for name, passed in FINAL_CHECKS.items() if not passed
    ]
    validation_memory = cuda_memory_snapshot('after-final-validation')
    FINAL_RESULT = {
        'source_commit': canonical_commit,
        'seconds': final_step_seconds,
        'optimizer_step_counts': dict(optimizer_step_counts),
        'optimizer_callback': optimizer_step_records,
        'gradient_tensors': len(final_gradients),
        'optimizer_parameter_states': len(final_optimizer.state),
        'adam_state_steps': optimizer_state_steps,
        'adam_moment_tensors': len(moment_tensors),
        'tracked_parameter': TRACKED_PARAMETER_NAME,
        'tracked_parameter_index': TRACKED_PARAMETER_INDEX,
        'parameter_before': final_parameter_before,
        'parameter_after': final_parameter_after,
        'parameter_delta': parameter_delta,
        'checks': FINAL_CHECKS,
        'cuda_memory': optimizer_step_memory,
        'validation_cuda_memory': validation_memory,
    }
    REPORT['final_step'] = FINAL_RESULT
    REPORT['memory']['optimizer_step_complete'] = optimizer_step_memory
    REPORT['memory']['after_final_validation'] = validation_memory
    if failed_final_checks:
        raise RuntimeError(f'Final checks failed: {failed_final_checks}')

    FINAL_STEP_COMPLETED = True
    REPORT['status'] = 'PASS'
    REPORT.pop('failed_stage', None)
    REPORT.pop('error', None)
    REPORT['completed_at_utc'] = datetime.now(timezone.utc).isoformat()
    save_report()
except Exception as error:
    REPORT['optimizer_step_counts'] = dict(optimizer_step_counts)
    failure_stage = (
        'optimizer-step-validation'
        if optimizer_step_counts['completed']
        else 'optimizer-step'
    )
    record_failure(failure_stage, error)
    raise
finally:
    step_pre_hook.remove()
    step_post_hook.remove()

print('PASS: one complete H200 optimizer step satisfied every contract')
FINAL_RESULT


## 12. Read the durable verdict

A PASS establishes this one-step checkpoint/runtime contract only. It is not a learning result, throughput benchmark, or proof that the path fits an 80 GB GPU. Cold-load and first-step timings are diagnostic.


In [ ]:
durable_report = json.loads(REPORT_PATH.read_text(encoding='utf-8'))
verdict = {
    'status': durable_report['status'],
    'run_id': durable_report['run_id'],
    'source_commit': durable_report['source']['resolved_commit'],
    'gpu': durable_report['environment']['gpu_name'],
    'model_load_seconds': durable_report['models']['load_seconds'],
    'final_step_seconds': durable_report.get('final_step', {}).get('seconds'),
    'peak_allocated_gib': durable_report.get('final_step', {})
    .get('cuda_memory', {})
    .get('peak_allocated_gib'),
    'report_path': str(REPORT_PATH),
}
if durable_report['status'] != 'PASS':
    raise AssertionError(f'Preflight did not pass; inspect {REPORT_PATH}')
verdict


## Development loop and next decision

For another compatible forward or loop experiment: change locally, commit and push, replace SOURCE_COMMIT with the new full SHA, rerun source sync and hot-path reload, then rerun the preview with unchanged weights.

When the preview is accepted, restart the H200 kernel and run top-to-bottom for a canonical one-commit report. Model-class, loader-precision, configuration-contract, or input-dataclass changes always require that fresh run.

Stop the H200 kernel after copying the report path. Source and synthetic data may disappear; model artifacts and reports remain in the Volume.
